# Dynare: RBC with Variable Capital Utilization

In this lab we will work through an RBC model with variable capital utilization. The objective of this lab is to show how one 
+ writes down the model on paper
+ solves for the steady state using matlab
+ solves for the (approximate) policy functions in Dynare
+ uses the policy function from Dynare to conduct simulations in matlab

By the end of the lab, you should be comfortable with all of these tasks.

## Step 1: Write down the model

The first thing to do is write down our optimization problem. Let $x_t$ denote capital utilization at time $t$. Then our optimization problem is 

\begin{align}
    & \max_{\left\{c_t,k_{t},n_t,x_t\right\}_{t=1}^{\infty}} \sum_{t=1}^{\infty} \beta^t \left(\frac{c_t^{1-\gamma}-1}{1-\gamma}- \psi \frac{n_t^{1+\varphi}}{1+\varphi}\right) \\
    k_{t}=&e^{z_t}\left(x_t k_{t-1}\right)^{\alpha} n_t^{1-\alpha}+(1-\Delta\left(x_t\right))k_{t-1}-c_t \\
    \Delta\left(x_t\right)=&\delta x_t^{1+\xi} \\
    z_t =& \rho z_{t-1} + \epsilon_t & \epsilon_t \sim \mathcal{N}\left(0,\sigma^2\right)  \\
    k_0 > 0 \text{ given}
\end{align}

Note that I am using the timing convention that Dynare uses. That is, all choices are time $t$ variables. The next thing we need to do is set up the Lagrangian. 

\begin{align}
    \mathcal{L}=&...+\beta^{t}\left\{\frac{c_t^{1-\gamma}-1}{1-\gamma}- \psi \frac{n_t^{1+\varphi}}{1+\varphi} + \lambda_t\left[e^{z_t}\left(x_t k_{t-1}\right)^{\alpha} n_t^{1-\alpha}+(1-\delta x_t^{1+\xi})k_{t-1}-c_t-k_t\right]\right\}+ \\
    & \beta^{t+1}\left\{\frac{c_{t+1}^{1-\gamma}-1}{1-\gamma}- \psi \frac{n_{t+1}^{1+\varphi}}{1+\varphi} + \lambda_{t+1}\left[e^{z_{t+1}}\left(x_{t+1} k_{t}\right)^{\alpha} n_{t+1}^{1-\alpha}+(1-\delta x_{t+1}^{1+\xi})k_{t}-c_{t+1}-k_{t+1}\right]\right\}+...
\end{align}

Now, as usual, we take first order conditions.
\begin{align}
    \frac{\partial \mathcal{L}}{\partial c_t}:& \beta^t\left(c_t^{-\gamma}-\lambda_t\right)=0 \\
    \frac{\partial \mathcal{L}}{\partial n_t}:& \beta^t\left(\psi n_t^{\varphi}-\lambda_t\left(1-\alpha\right)e^{z_t}\left(x_t k_{t-1}\right)^{\alpha} n_t^{-\alpha} \right)=0 \\
    \frac{\partial \mathcal{L}}{\partial x_t}:& \beta^t\left(\alpha e^{z_t} x_t^{\alpha-1}k_{t-1}^{\alpha}n_t^{1-\alpha}-\delta (1+\xi) x_t^{\xi} k_{t-1}\right)=0 \\
    \frac{\partial \mathcal{L}}{\partial k_t}:& -\beta^t\lambda_t+\beta^{t+1}\lambda_{t+1}\left(\alpha e^{z_t} x_t^{\alpha}k_{t-1}^{\alpha-1}n_t^{1-\alpha}+\left(1-\delta x_t^{1+\xi}\right)\right) \\
    \frac{\partial \mathcal{L}}{\partial \lambda_t}:& \beta^t\left(e^{z_t}\left(x_t k_{t-1}\right)^{\alpha} n_t^{1-\alpha}+(1-\delta x_t^{1+\xi})k_{t-1}-c_t-k_t\right)
\end{align}

At this point, I have 5 equations and 5 unknown endogenous variables (remember that $z_t$ is exogenous). I could simplify some equations first before trying to find the steady state. However, since I'm going to use a numerical solver, I will just leave them as is.

## Step 2: Find the steady state.

The next thing we will do is solve for the steady state numerically. The first thing to do is to write the function we want to find the zeros of. That is, we need to define our function $H$ where $H(Y^{*})=0$. As I mentioned in the previous lab, instead of specifying all parameters and solving for the steady state of the variables, I am going to impose that $n^*=\frac{1}{3}$ and then treat $\psi$ as a variable. Therefore, I am going to define $Y$ as 
\begin{align}
Y\equiv \begin{bmatrix}
    c \\
    \psi \\
    k \\
    \lambda \\
    x
\end{bmatrix}
\end{align}

Now we can write down the function that we are going to find the zeros of using $\mathtt{fsolve}$

In [ ]:
function err=H(Y,mp)
    c=Y(1)
    psi=Y(2)
    k=Y(3)
    lda=Y(4)
    x=Y(5)
    gamma=mp.gamma;
    delta=mp.delta;
    xi=mp.xi;
    n=mp.n0;
    alpha=mp.alpha;
    vphi=mp.vphi;
    beta=mp.beta;
    

    err(1)=c^(-gamma)-lda;
    err(2)=psi*n^(vphi)-lda*(1.0-alpha)*(x*k)^(alpha)*n^(-alpha);
    err(3)=-lda+beta*lda*(alpha*x^alpha*k^(alpha-1)*n^(1.0-alpha)+(1.0-delta*x^(1+xi)));
    err(4)=(x*k)^(alpha)*n^(1.0-alpha)+(1.0-delta*x^(1+xi))*k-c-k;
    err(5)=alpha*x^(alpha-1.0)*k^(alpha)*n^(1.0-alpha)-delta*(1.0+xi)*x^xi*k;
end

Now we need to write the first block of our code that initializes parameters and specifies initial guesses for the steady state. 

In [ ]:
clc;
clear;

mp=set_model_params()
Y0=set_Y0(mp)
options = optimset('Display','iter','MaxFunEvals',1000000,'TolFun',1e-20,'MaxIter',10000);

[fsolve_x,fval]=fsolve(@(y) H(y,mp),Y0,options);

In [ ]:
function mp=set_model_parameters()
    mp={"alpha",1.0/3.0,....
        "beta",0.95,....
        "delta",0.1,....
        "gamma",1,....
        "vphi",2,....
        "xi",1,....
        "rho",0.9,....
        "sigma",0.02,...
        "n0", 1.0/3.0};  
end

In [ ]:
function Y0=set_Y0(mp)
    c0=.5;
    psi0=2.5;
    x0=1;
    lda0=c0^(-gamma);
    k0=(mp.alpha*mp.beta*mp.n0^(1.0-mp.alpha))^(1.0/(1.0-mp.alpha));
    Y0=[c0; psi0; k0; lda0];
end

Once $\mathtt{fsolve}$ finds the steady state, the next thing we want to do is save the parameters and steady state values in a $\mathtt{.txt}$ file so we can load everything into our Dynare file. 

In [ ]:
steady_state_names=["z";"k";"c";"n";"x"];
steady_state_vals=[0.0; fsolve_x(1); fsolve_x(3); mp.n0; fsolve_x(5)];
writematrix(strcat(strcat(steady_state_names,"="),strcat(string(steady_state_vals),";")),"steady_state.txt");


params_names=["beta";"alpha";"delta";"psi";"vphi";"gamma";"rho"; "sigma"; "xi"];
params_vals=[mp.beta;mp.alpha;mp.delta;fsolve_x(2);mp.vphi;mp.gamma;mp.rho;mp.sigma;mp.xi];
writematrix(strcat(strcat(params_names,"="),strcat(string(params_vals),";")),"params.txt");

We'll use the $\mathtt{writematrix}$ and $\mathtt{strcat}$ functions to save the steady state values as $\mathtt{.txt}$ files.

We need each row of the $\mathtt{steady\_state.txt}$ file to be of the form $\mathtt{var\_name=value;}$. Therefore, we use multiple calls to $\mathtt{strcat}$ to write the entries in this format. First we concatenate the variable name with the equals sign. Then we concatenate the variable value (as a string) with a semi-colon. Finally, we concatenate the return of our first call to $\mathtt{strcat}$ with the return of our second call to $\mathtt{strcat}$ to obtain the final result. We then use $\mathtt{writematrix}$ to save this. The second argument of $\mathtt{writematrix}$ is the name of the file where things will be saved.

## Step 3: Dynare

The Dynare file is made up of blocks. The first block is the variable declaration block. In the variable declaration block, we declare all of our variables. If a variable is endogenous, we use the $\mathtt{var}$. If the variable is exogenous, we use $\mathtt{varexo}$. Note that every line in the Dynare file ends with a semi-colon.

In [ ]:
var z k c n x;
varexo e;

After declaring our variables, we need to declare the parameters that are used in the model. To declare parameters, we use $\mathtt{parameters}$. After declaring the parameters, we need to assign values to the parameters. We can assign values explicitly like I've done below, or we can load the values from a file. You should avoid the first approach in general.

In [ ]:
parameters beta alpha delta psi vphi gamma rho sigma xi;
beta=0.95;
alpha=1.0/3.0;
delta=0.1;
psi=21.6;
vphi=2.0;
gamma=1.0;
rho=0.9;
sigma=0.02;
xi=1.0;

The next thing we need to do is declare our model. We do so using the $\mathtt{model}$ block. In the $\mathtt{model}$ block, you write your equations just as you would on paper. Predetermined variables are denoted by $(-1)$. Future variables are denoted by $(+1)$. For example, start of period capital is predetermined. Therefore, we would write this as $k(-1)$. In the Euler equation for capital, we have future consumption. Therefore, we would write this as $c(+1)$.

In [ ]:
model;
    c^(-gamma)=beta*(c(+1)^(-gamma))*(alpha*exp(z(+1))*x(+1)^alpha*(k)^(alpha-1.0)*n(+1)^(1.0-alpha)+(1.0-delta*x(+1)^(1.0+xi)));
    psi*n^(vphi)=(1.0-alpha)*exp(z)*(x*k(-1))^(alpha)*n^(-alpha)*c^(-gamma);
    k=exp(z)*(x*k(-1))^alpha*n^(1.0-alpha)+(1.0-delta*x^(1+xi))*k(-1)-c;
    alpha*exp(z)*x^(alpha-1)*k(-1)^alpha*n^(1-alpha)=k(-1)*delta*(1.0+xi)*x^(xi);
    z=rho*z(-1)+e;
end;

Since Dynare uses perturbation, we need to provide a point around which we should expand the policy functions. Dynare uses the steady state as this point. For the next block, we have two options. We can use $\mathtt{initval}$ or $\mathtt{steady\_state\_model}$. If we use $\mathtt{initval}$, we provide an initial guess for steady state, and then Dynare tries to solve for the steady state. If we use $\mathtt{steady\_state\_model}$, Dynare checks to see if our provided values are a steady state of the model. If they are not, then Dynare returns an error. My recommendation is to solve for the steady state outside of Dynare, and then use the $\mathtt{steady_state_model}$ block. If Dynare states that the values you provided are not steady state values, then you should check the equations in both the function you used to solve for the steady state and the equations in the $\mathtt{model}$ block. Most likely you have an incorrect equation somewhere. Just as we can do when assigning values to parameters, for steady state values we can either declare the values explicitly in the Dynare file or load them from an external file. Again, loading them from an external file is a better approach.

In [ ]:
steady_state_model;
\\initval;
    z=0;
    k=0.42103;
    c=1.5999;
    n=0.33333;
    x=0.72548;
end;

We can also use Dynare to compute impulse responses and stochastic simulations. In both cases, we need to tell Dynare which innovations we want to study the response of. To do this, we use the $\mathtt{shocks}$ block. In the $\mathtt{shocks}$ block, we tell Dynare which exogenous variable we want to consider by using $\mathtt{var}$. We use $\mathtt{stderr}$ to tell Dynare the standard deviation of the innovation. By default, the impulse response considers a one standard deviation innovation.

In [ ]:
shocks;
    var e; stderr sigma;
end;

Finally, we need to tell Dynare which perturbation order we want to consider.

In [ ]:
stoch_simul(order=1);

Now we can put everything together. Note that when initializing parameters, I use the macro $\mathtt{\#include}$ to load the parameters from the file $\mathtt{params.txt}$.

In [ ]:
var z k c n x;
varexo e;

parameters beta alpha delta psi vphi gamma rho sigma xi;
@#include "params.txt"


model;
    c^(-gamma)=beta*(c(+1)^(-gamma))*(alpha*exp(z(+1))*x(+1)^alpha*(k)^(alpha-1.0)*n(+1)^(1.0-alpha)+(1.0-delta*x(+1)^(1.0+xi)));
    psi*n^(vphi)=(1.0-alpha)*exp(z)*(x*k(-1))^(alpha)*n^(-alpha)*c^(-gamma);
    k=exp(z)*(x*k(-1))^alpha*n^(1.0-alpha)+(1.0-delta*x^(1+xi))*k(-1)-c;
    alpha*exp(z)*x^(alpha-1)*k(-1)^alpha*n^(1-alpha)=k(-1)*delta*(1.0+xi)*x^(xi);
    z=rho*z(-1)+e;
end;

steady_state_model;
\\initval;
    @#include "steady_state.txt"
end;

shocks;
var e; stderr sigma;
end;

stoch_simul(order=1);



## Step 4: Simulation outside of Dynare

The last thing we will do is simulate our model. Recall that Dynare constructs approximations of the policy functions around steady state. Using these policy functions we can construct a time series for the variables. The first thing we need to so is extract the approximated decision rules from the Dynare output. Recall that the output is stored in the structure **oo_**. Just as we did last time, we'll start by creating an index for the variables. A variable's index number corresponds to its order in the var declaration in Dynare.

In [ ]:
idx_z=1;
idx_k=2;
idx_c=3;
idx_n=4;
idx_x=5;

Remember, the steady state values are in the same order as we declared the variables. However, the policy functions are not. To find out what row of the policy function corresponds to a given variable, we need to look at **oo_.dr.inv_order_val**. Since we declared **z** as the first variable, **oo_.dr.inv_order_val(1)** will tell us which row of the policy function matrix corresponds to **z**. 

Next we need to split our variables into state and control variables. In last week's notation, state variables were denoted as $y_t$ and controls were $x_t$. This leads to the following representation of the system.

\begin{align}
y_t=Ay_{t-1}+B\epsilon_t \\
x_t=Cy_{t-1}+D\epsilon_t
\end{align}

We'll start by constructing $A$ and $B$. Our two state variables are $k$ and $z$. Therefore, $A$ and $B$ will use include the policy functions for $k$ and $z$. Recall that the **oo_.dr.ghx** contains the coefficients on the variables while **oo_.dr.ghu** contains the coefficients on the innovations ($\epsilon$'s). 

# NOTE: 
Make sure to put the $A$ matrix in the correct ordering. That is, if state variable $x_j$ comes before state variable $x_k$ in **oo_.dr.inv_order_var**, then the row corresponding to $x_j$ must come before the row corresponding to $x_k$ in the matricies $A$ and $B$.

In [ ]:
A=[oo_.dr.ghx(oo_.dr.inv_order_var(idx_k),:);...
   oo_.dr.ghx(oo_.dr.inv_order_var(idx_z),:)]

B=[oo_.dr.ghu(oo_.dr.inv_order_var(idx_k),:);...
   oo_.dr.ghu(oo_.dr.inv_order_var(idx_z),:)]

Now we need to construct the matricies for out controls. Our controls are $c,n,x$.

In [ ]:
C=[oo_.dr.ghx(oo_.dr.inv_order_var(idx_c),:);...
   oo_.dr.ghx(oo_.dr.inv_order_var(idx_n),:);...
   oo_.dr.ghx(oo_.dr.inv_order_var(idx_x),:)]

D=[oo_.dr.ghu(oo_.dr.inv_order_var(idx_c),:);...
   oo_.dr.ghu(oo_.dr.inv_order_var(idx_n),:);...
   oo_.dr.ghu(oo_.dr.inv_order_var(idx_x),:)]

Now that we have constructed our matricies, we can conduct our simulation. We first need to draw a sequence of $\epsilon$'s. Let's consider an $N$ period simulation. We will use matlab's $\mathtt{randn}$ function to draw $N$ realizations of epsilon.

In [ ]:
N=10000;
epsi=randn(1,N)*sigma;

Now we need to write a $\mathtt{for}$ loop to go through the simulation. I will let $U$ be the vector of states and $V$ the vector of controls. Both are in deviation from steady state. At the end of the simulation we will add back in the steady state values to obtain the variables in levels. Remember, the steady state values are stored in **oo_.dr.ys** and are stored in the order in which you declared the variables.

In [ ]:
U(:,1)=B*epsi(1);
V(:,1)=D*epsi(1);

for j=2:N
    U(:,j)=A*U(:,j-1)+B*epsi(j);
    V(:,j)=C*U(:,j-1)+D*epsi(j);
end

Uss=[oo_.dr.ys(idx_k); oo_.dr.ys(idx_z)];
Vss=[oo_.dr.ys(idx_c); oo_.dr.ys(idx_n); oo_.dr.ys(idx_x)];
U=U+Uss*ones(1,N);
V=V+Vss*ones(1,N);

Finally, we can plot the results.

In [ ]:
figure()
subplot(3,2,1)
plot(1:N,U(1,:),1:N,ones(1,N)*Uss(1),'linewidth',3)
title('Captial')
subplot(3,2,2)
plot(1:N,U(2,:),1:N,ones(1,N)*Uss(2),'linewidth',3)
title('Productivity')
subplot(3,2,3)
plot(1:N,V(1,:),1:N,ones(1,N)*Vss(1),'linewidth',3)
title('Consumption')
subplot(3,2,4)
plot(1:N,V(2,:),1:N,ones(1,N)*Vss(2),'linewidth',3)
title('Labor')
subplot(3,2,5)
plot(1:N,V(3,:),1:N,ones(1,N)*Vss(3),'linewidth',3)
title('Captial Utilization')

figure()
subplot(3,2,1)
histogram(U(1,:))
title('Captial')
subplot(3,2,2)
histogram(U(2,:))
title('Log Productivity')
subplot(3,2,3)
histogram(V(1,:))
title('Consumption')
subplot(3,2,4)
histogram(V(2,:))
title('Labor')
subplot(3,2,5)
histogram(V(3,:))
title('Captial Utilization')

bc_vars=[U(1,:); U(2,:); V(1,:); V(2,:); V(3,:)]';
corr(bc_vars)